In [7]:
import json
import openai

In [8]:
with open("../data/courses.json", "r") as f:
    courses = json.load(f)
with open("../data/glossary.json", "r") as f:
    glossary = json.load(f)

In [9]:
gcodes = glossary.keys()
lcodes = [c["split_course_code"][0] for c in courses]
ccodes = {c["split_course_code"][0] for c in courses}

In [46]:
diff = ccodes - gcodes
print(len(diff))
buffer = []
for s in diff:
    buffer.append((s, lcodes.count(s)))
print(sorted(buffer, key=lambda x: x[1], reverse=True))

80
[('RSM', 98), ('HMB', 63), ('VIC', 56), ('TRN', 46), ('ESS', 37), ('BMS', 35), ('DRM', 35), ('MUN', 35), ('NML', 33), ('INS', 30), ('CDN', 26), ('PHS', 24), ('CRE', 24), ('MGY', 24), ('CHC', 23), ('AFR', 22), ('CSE', 22), ('ACT', 21), ('CLT', 20), ('NEW', 18), ('FSL', 18), ('EDS', 16), ('DTS', 16), ('COG', 16), ('LCT', 15), ('URB', 15), ('CAR', 15), ('CJS', 15), ('SMC', 14), ('PHC', 14), ('APM', 12), ('REN', 12), ('PRT', 11), ('UNI', 11), ('DHU', 9), ('MCS', 9), ('EST', 9), ('WRR', 9), ('ARH', 9), ('INT', 9), ('BIO', 9), ('FIN', 8), ('IRW', 8), ('CAS', 8), ('ABP', 8), ('PDC', 8), ('BPM', 7), ('FCS', 7), ('JEG', 7), ('ANA', 6), ('AMS', 6), ('MGR', 6), ('LAS', 6), ('INI', 5), ('EUR', 5), ('STS', 4), ('WDW', 4), ('BCB', 4), ('JGU', 4), ('ENT', 3), ('JLS', 3), ('JSN', 3), ('IFP', 2), ('JFL', 2), ('CTA', 2), ('JSU', 2), ('JIG', 2), ('JCR', 1), ('JGA', 1), ('JSM', 1), ('ETH', 1), ('JQR', 1), ('MHB', 1), ('MIJ', 1), ('JHU', 1), ('JWE', 1), ('EHJ', 1), ('JFP', 1), ('JWB', 1), ('CJH', 1)]


In [50]:
def get_examples(prefix, limit=20):
    examples = []
    for c in courses:
        if c["split_course_code"][0] == prefix:
            examples.append(f"{c["course_code"]} {c["title"]}")
        if len(examples) > 20:
            break
    return examples

In [34]:
client = openai.OpenAI(base_url="http://localhost:1337/v1", api_key="helloworld")

In [43]:
task = """
PREFIX {prefix}

EXAMPLES
{courses}
"""

In [51]:
instructions = """
You are a tool part of a processing pipeline for University of Toronto courses.
Your task is to guess the meaning of 3 letter course codes. 
They are often an acronym or the starting letters of their department / name.
A list of example courses is provided.
Output just your best guess only, with no extra text or explanation.
"""

In [52]:
def send_loop(prefix):
    m = task.format(prefix=prefix, courses=str(get_examples(prefix)))
    conversation = [
        {"role": "system", "content": ""},
        {"role": "user", "content": "PREFIX MAT"},
        {"role": "assistant", "content": "Mathematics"},
        {"role": "user", "content": m}
    ]
    print(conversation)
    response = client.chat.completions.create(
        model="Jan-v3.5-4B-Q4_K_XL",
        messages=conversation,       # conversation history
        max_tokens=20,      # cap response length
        temperature=0.3,      # 0 = deterministic, 2 = very random
        stream=False,         # True to receive tokens as they generate
    )
    guess = response.choices[0].message.content
    return guess

In [54]:
for prefix, c in buffer:
    guess = send_loop(prefix)
    print(prefix, c, guess)

[{'role': 'system', 'content': ''}, {'role': 'user', 'content': 'PREFIX MAT'}, {'role': 'assistant', 'content': 'Mathematics'}, {'role': 'user', 'content': "\nPREFIX ENT\n\nEXAMPLES\n['ENT200H1 Entrepreneurial Strategy', 'ENT310H1 Innovation Strategy and Intellectual Property for Entrepreneurs', 'ENT391H1 Exploring New Ventures']\n"}]
ENT 3 Entrepreneurship
[{'role': 'system', 'content': ''}, {'role': 'user', 'content': 'PREFIX MAT'}, {'role': 'assistant', 'content': 'Mathematics'}, {'role': 'user', 'content': "\nPREFIX LCT\n\nEXAMPLES\n['LCT202Y1 Forms of Representation', 'LCT203H1 Empires I', 'LCT204H1 Canons and Canonicity', 'LCT205H1 Empires II', 'LCT301H1 Critical Writing Seminar', 'LCT302H1 Pasts and Futures', 'LCT306H1 Culture and Media', 'LCT307H1 Periodization and Cultural History', 'LCT308H1 Identities', 'LCT349H1 Special Topics in Literature and Critical Theory', 'LCT401H1 Seminar in Comparative Literature', 'LCT402H1 Translation and Comparativity', 'LCT403H1 Advanced Topics